# PCFG v10.1 — Experimental notebook

## Reading guide

This notebook implements the current synthetic testbed for the sharing/separation cost model. The derivation is assumed as given; the comments here focus on the **mapping from the theory to the implementation, the controls, and the places where the present operationalization is stronger or weaker than the theoretical ideal**.

The main empirical objects are the capacity curves and matched-language contrasts. In particular:
- **C vs Cτ** is the current τ-style manipulation: same class table/grammar, differing in whether a local determiner cue exposes gender.
- **D vs Dlo** is intended as an ε manipulation, but the current Hamming construction can simultaneously alter the difficulty of the native lexical lookup, so it is not yet a clean ε-only intervention.
- **D vs Dpost** probes whether moving agreement farther from its licensing information changes the capacity profile.

The code is deliberately split into latent language specification → surface realization → PCFG sampling → leakage checks → model → evaluation → sweep. That separation is the main thing to audit when interpreting the results.


## 1. Environment and execution

Infrastructure only. The relevant scientific choices begin in the language specification below. Runtime settings are kept separate so CUDA/AMP/caching changes do not alter the synthetic data-generating process.


In [ ]:
import os
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import math, random, time, json, itertools
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Set, Any
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 16
print(f"Device: {device}  |  NUM_WORKERS: {NUM_WORKERS}  (cpu_count={os.cpu_count()})")


# ══════════════════════════════════════════════════════════════════════


## Why 1024 shared stems?

The noun inventory is intentionally large relative to the tiny token-identity bottleneck. Each `(language, stem)` pair has a deterministic latent class, so the model faces a finite lexical-state problem rather than a compositional rule that can be solved from the stem string itself.

This is also important for the capacity manipulation: `n_embd_tok` is meant to constrain how much noun identity can be retained before the model projects into the wider residual stream. The 1024-stem inventory therefore makes the lexical lookup problem nontrivial while remaining exactly reproducible.


## 2. Synthetic languages and latent class systems

This is the experimental-design table. The important point is not the individual linguistic plausibility of each language, but which dimensions are held invariant across each comparison.

- **A** is the anchor/reference class table.
- **B** is a relatively close 8-way perturbation of A.
- **C/Cτ** share the same class system and structural grammar; the intervention is whether the determiner exposes gender.
- **D/Dlo** share the same realization and structural grammar; the intervention is the anchor-table Hamming distance. This is intended to vary shared-route information, but it also changes the native noun→class mapping, which is why it cannot yet be identified with ε alone.
- **D/Dpost** keep the language specification fixed except for adjective timing, isolating a structural-distance effect.
- **E** uses an independent 6-way class system and is therefore a qualitatively different control rather than a member of the 8-way Hamming continuum.

The executable `LANG_SPEC` is the authoritative specification. Note that B is actually `hamming=154`; an old comment mentioning 64 is stale bookkeeping, not an experimental condition.


In [ ]:
N_NOUN_STEMS  = 1024
N_ADJ_LEMMAS  = 8
N_VTR_LEMMAS  = 6
N_VINTR_LEMMAS = 4
N_PREPS       = 4
N_VTR_LEMMAS  = 6
N_VINTR_LEMMAS = 4
N_PREPS       = 4

GENDER_SYSTEMS = {
    6: [f'N{i}' for i in range(1, 7)],
    8: [f'G{i}' for i in range(1, 9)],
}

# Core experiment: A/B/C/Ctau/D/Dlo/Dpost all use the same 8-way class
# alphabet. E remains an explicitly independent 6-way control rather than
# being folded into the main epsilon comparisons.
LANG_SPEC: Dict[str, Dict[str, Any]] = {
    'A': dict(anchor=True,  word_order='SVO', adj_pos='post',
              gender_system=8, det_marks_gender=True,  det_fuses_number=False,
              case_marking=False, verb_agree=('number',),          verb_agree_target='subject',
              rel_agrees=True,  hamming=0),

    'B': dict(anchor=False, word_order='SVO', adj_pos='post',
              gender_system=8, det_marks_gender=True,  det_fuses_number=True,
              case_marking=False, verb_agree=('number', 'gender'), verb_agree_target='subject',
              rel_agrees=True,  hamming=154),

    'C': dict(anchor=False, word_order='SOV', adj_pos='post',
              gender_system=8, det_marks_gender=False, det_fuses_number=False,
              case_marking=True, verb_agree=(),                   verb_agree_target=None,
              rel_agrees=False, hamming=512),

    'Ctau': dict(anchor=False, word_order='SOV', adj_pos='post',
              gender_system=8, det_marks_gender=True, det_fuses_number=False,
              case_marking=True, verb_agree=(),                   verb_agree_target=None,
              rel_agrees=False, hamming=512),

    # D vs Dlo is the primary lexical-epsilon twin: same grammar and realization,
    # invariant determiner, and shared agreement-output channel; only the
    # noun->class codebook differs in hamming distance.
    'D': dict(anchor=False, word_order='SOV', adj_pos='post', adj_timing='early',
              gender_system=8, det_marks_gender=False, det_fuses_number=False,
              case_marking=True, verb_agree=('number', 'gender'), verb_agree_target='object',
              rel_agrees=False, hamming=870),

    'Dlo': dict(anchor=False, word_order='SOV', adj_pos='post', adj_timing='early',
              gender_system=8, det_marks_gender=False, det_fuses_number=False,
              case_marking=True, verb_agree=('number', 'gender'), verb_agree_target='object',
              rel_agrees=False, hamming=154),

    # Structural control: identical to D except agreement timing. The same
    # sampled modifier can either follow the adjective (early) or intervene
    # before it (late), creating a controlled memory-distance manipulation.
    'Dpost': dict(anchor=False, word_order='SOV', adj_pos='post', adj_timing='late',
              gender_system=8, det_marks_gender=False, det_fuses_number=False,
              case_marking=True, verb_agree=('number', 'gender'), verb_agree_target='object',
              rel_agrees=False, hamming=870),

    'E': dict(anchor=False, word_order='SVO', adj_pos='post',
              gender_system=6, det_marks_gender=True, det_fuses_number=False,
              case_marking=False, verb_agree=('number', 'gender'), verb_agree_target='subject',
              rel_agrees=True, independent_gender=True, hamming='incommensurable'),
}
LANGS = list(LANG_SPEC.keys())

TWIN_PAIRS = [
    ('D', 'Dlo',   'epsilon (lexical class distance: hamming 870 vs 154, out of 1024)'),
    ('C', 'Ctau',  'tau (DET marks gender: no vs yes; delayed post-nominal agreement)'),
    ('D', 'Dpost', 'structural epsilon (agreement timing: immediately post-nominal vs after modifier chain)'),
    ('A', 'B',     'epsilon, low end (hamming 0 vs 64), naturalistic pool'),
]

def genders_for(lang):
    return GENDER_SYSTEMS[LANG_SPEC[lang]['gender_system']]

# ── Gender assignment tables ──
# Deterministic per-language seeds: avoid Python's process-salted hash(), which
# made old runs reproducible only within a single interpreter lifetime.
TABLE_SEEDS = {
    'A': 1001, 'B': 1002, 'C': 1003, 'Ctau': 1004,
    'D': 1005, 'Dlo': 1006, 'Dpost': 1007, 'E': 1008,
}

NOUN_STEMS = [f"n{idx:04d}" for idx in range(N_NOUN_STEMS)]

def build_anchor_table(seed=0):
    rng = random.Random(seed)
    classes = GENDER_SYSTEMS[LANG_SPEC['A']['gender_system']]
    return {stem: rng.choice(classes) for stem in NOUN_STEMS}

ANCHOR_TABLE = build_anchor_table(seed=0)

def build_gender_table(lang):
    spec = LANG_SPEC[lang]
    classes = GENDER_SYSTEMS[spec['gender_system']]
    rng = random.Random(TABLE_SEEDS[lang])
    if spec.get('independent_gender'):
        return {stem: rng.choice(classes) for stem in NOUN_STEMS}
    if spec.get('anchor'):
        return dict(ANCHOR_TABLE)
    h = spec['hamming']
    flip_stems = set(rng.sample(NOUN_STEMS, h))
    table = dict(ANCHOR_TABLE)
    for stem in flip_stems:
        current = table[stem]
        table[stem] = rng.choice([c for c in classes if c != current])
    return table

GENDER_TABLE: Dict[str, Dict[str, str]] = {L: build_gender_table(L) for L in LANGS}

for L in LANGS:
    spec = LANG_SPEC[L]
    if not spec.get('anchor') and not spec.get('independent_gender'):
        realized = sum(1 for s in NOUN_STEMS if GENDER_TABLE[L][s] != ANCHOR_TABLE[s])
        assert realized == spec['hamming'], (L, realized, spec['hamming'])
print("Gender tables built and Hamming distances verified against spec.")


def assert_twin_invariants():
    fields = ['word_order', 'adj_pos', 'adj_timing', 'gender_system',
              'det_marks_gender', 'det_fuses_number', 'case_marking',
              'verb_agree', 'verb_agree_target', 'rel_agrees']
    allowed = {
        ('D', 'Dlo'): {'hamming'},
        ('C', 'Ctau'): {'det_marks_gender'},
        ('D', 'Dpost'): {'adj_timing'},
    }
    for a, b in allowed:
        sa, sb = LANG_SPEC[a], LANG_SPEC[b]
        diffs = {f for f in fields if sa.get(f) != sb.get(f)}
        assert diffs <= allowed[(a, b)], (a, b, diffs)
    print("Twin-language invariants verified.")

assert_twin_invariants()


# ══════════════════════════════════════════════════════════════════════


## Design constraint: shared agreement inventory

Agreement-bearing surface symbols are shared across the 8-way core languages. This is a deliberate restriction on the output vocabulary: otherwise a cross-language accuracy difference could simply reflect relearning separate inflectional lexica rather than a difference in how the latent agreement feature is represented.

Language-specific function words remain language-specific where required by the realization specification. Thus the manipulation is not “all languages use the same tokens”; it is specifically that the **agreement dimension has a common surface inventory**.


## 3. Surface realization

This layer is the interface between the latent specification and the observed token sequence. It should be read together with the PCFG below: this code realizes a structure and feature assignment; it does not determine the structural probabilities.

The key audit question here is whether the surface vocabulary accidentally gives the model a shortcut that was not part of the intended condition. In particular, shared agreement symbols make the agreement target comparable across languages, while language IDs/function words still permit the model to condition on the language when that is genuinely available.


In [ ]:
DOT = '.'
BOS, EOS, PAD = '<BOS>', '<EOS>', '<PAD>'
LANG_TOK = {L: f'<L{L}>' for L in LANGS}

BARE_NOUNS = list(NOUN_STEMS)
CASE_NOUNS = {(stem, case): f"{stem}_{case}" for stem in NOUN_STEMS for case in ('NOM', 'ACC')}


def noun_surface(lang, stem, role):
    if LANG_SPEC[lang]['case_marking']:
        case = 'NOM' if role == 'subj' else 'ACC'
        return CASE_NOUNS[(stem, case)]
    return stem

DETS: Dict[str, Any] = {}
NUMS: Dict[str, Dict[str, str]] = {}
ADJS: Dict[str, Dict[str, List[str]]] = {}
VTR: Dict[str, Any] = {}
VINTR: Dict[str, Any] = {}
RPROS: Dict[str, Any] = {}
PREPS: Dict[str, List[str]] = {}
CONJS: Dict[str, str] = {}

# Shared 8-way agreement inventories.
SHARED_CLASSES = GENDER_SYSTEMS[8]
ADJS_SHARED = {g: [f"agr_adj_{g}_{i}" for i in range(N_ADJ_LEMMAS)] for g in SHARED_CLASSES}
VTR_GENDER_SHARED = {
    (g, n): [f"agr_v_{g}_{n}_{i}" for i in range(N_VTR_LEMMAS)]
    for g in SHARED_CLASSES for n in ('SG', 'PL')
}
VINTR_GENDER_SHARED = {
    (g, n): [f"agr_vi_{g}_{n}_{i}" for i in range(N_VINTR_LEMMAS)]
    for g in SHARED_CLASSES for n in ('SG', 'PL')
}
RPRO_SHARED = {g: f"agr_rp_{g}" for g in SHARED_CLASSES}

for L in LANGS:
    spec = LANG_SPEC[L]
    genders = genders_for(L)

    if spec['det_marks_gender'] and spec['det_fuses_number']:
        DETS[L] = {(g, n): f"dt_{L}_{g}_{n}" for g in genders for n in ('SG', 'PL')}
    elif spec['det_marks_gender']:
        DETS[L] = {g: f"dt_{L}_{g}" for g in genders}
    else:
        DETS[L] = f"dt_{L}"

    if not spec['det_fuses_number']:
        NUMS[L] = {'SG': f"ns_{L}", 'PL': f"np_{L}"}

    # Shared agreement output for the 8-way core languages; E keeps an
    # independent output inventory because its six-way system is a control.
    if spec['gender_system'] == 8:
        ADJS[L] = ADJS_SHARED
    else:
        ADJS[L] = {g: [f"adj_{L}_{g}_{i}" for i in range(N_ADJ_LEMMAS)] for g in genders}

    feats = spec['verb_agree']
    if feats == ():
        VINTR[L] = {'__inv__': [f"v_{L}_inv_{i}" for i in range(N_VINTR_LEMMAS)]}
        VTR[L] = {'__inv__': [f"v_{L}_invtr_{i}" for i in range(N_VTR_LEMMAS)]}
    elif feats == ('number',):
        VINTR[L] = {n: [f"v_{L}_{n}_{i}" for i in range(N_VINTR_LEMMAS)] for n in ('SG', 'PL')}
        VTR[L] = {n: [f"v_{L}_{n}_{i}" for i in range(N_VTR_LEMMAS)] for n in ('SG', 'PL')}
    elif feats == ('number', 'gender'):
        if spec['gender_system'] == 8:
            VTR[L] = VTR_GENDER_SHARED
            VINTR[L] = VINTR_GENDER_SHARED
        else:
            VTR[L] = {(g, n): [f"v_{L}_{g}_{n}_{i}" for i in range(N_VTR_LEMMAS)]
                      for g in genders for n in ('SG', 'PL')}
            VINTR[L] = {(g, n): [f"v_{L}_{g}_{n}_{i}" for i in range(N_VINTR_LEMMAS)]
                        for g in genders for n in ('SG', 'PL')}
    else:
        raise ValueError(feats)

    if spec['rel_agrees']:
        RPROS[L] = RPRO_SHARED if spec['gender_system'] == 8 else {g: f"rp_{L}_{g}" for g in genders}
    else:
        RPROS[L] = f"rp_{L}"

    PREPS[L] = [f"p_{L}{i}" for i in range(N_PREPS)]
    CONJS[L] = f"cj_{L}"

ALL_TOKENS: Set[str] = set()
ALL_TOKENS.update(BARE_NOUNS)
ALL_TOKENS.update(CASE_NOUNS.values())
ALL_TOKENS.update([DOT, BOS, EOS, PAD])
ALL_TOKENS.update(LANG_TOK.values())

for L in LANGS:
    d = DETS[L]
    ALL_TOKENS.update(d.values() if isinstance(d, dict) else [d])
    if L in NUMS:
        ALL_TOKENS.update(NUMS[L].values())
    for g in genders_for(L):
        ALL_TOKENS.update(ADJS[L][g])
    for forms in VTR[L].values():
        ALL_TOKENS.update(forms)
    for forms in VINTR[L].values():
        ALL_TOKENS.update(forms)
    r = RPROS[L]
    ALL_TOKENS.update(r.values() if isinstance(r, dict) else [r])
    ALL_TOKENS.update(PREPS[L])
    ALL_TOKENS.add(CONJS[L])

VOCAB = sorted(ALL_TOKENS)
VOCAB_SIZE = len(VOCAB)
STOI = {w: i for i, w in enumerate(VOCAB)}
ITOS = {i: w for w, i in STOI.items()}
MAX_LEN = 192

print(f"Vocabulary: {VOCAB_SIZE} tokens  |  Languages: {LANGS}  |  Shared noun stems: {N_NOUN_STEMS}")


# ══════════════════════════════════════════════════════════════════════


## Why the sampler was rewritten

Earlier versions mixed explicit production probabilities with procedural draws (notably an additional PP draw after adjective realization). That made the nominal probabilities in the prose differ from the probabilities actually experienced by the model.

The current version removes that ambiguity: one `NP_HEAD` draw and one `MOD` draw determine the structural choices, with adjective timing affecting **where** the adjective is realized rather than whether an additional modifier is sampled. This is the reason the PCFG can now be audited independently of the realization code.


## 4. Explicit PCFG

The structural distribution is represented explicitly as normalized production families. The current rules are:

- `S`: single clause / coordination
- `CLAUSE`: intransitive / transitive
- `NP_HEAD`: adjective / no adjective
- `MOD`: none / PP / relative clause

PP and relative recursion are bounded. Importantly, these probabilities are **shared across languages**. Word order, case marking, agreement realization, and determiner behavior enter downstream through the language specification rather than through language-specific production probabilities.

This makes the grammar a controlled common substrate; the intended cross-language differences are in the mapping from the shared structural/latent variables to surface form.


In [ ]:
PCFG_RULES = {
    'S': [('single', 0.65), ('coord', 0.35)],
    'CLAUSE': [('intransitive', 0.35), ('transitive', 0.65)],
    'NP_HEAD': [('with_adj', 0.55), ('no_adj', 0.45)],
    'MOD': [('none', 0.45), ('pp', 0.35), ('rel', 0.20)],
}
MAX_PP_DEPTH = 1
MAX_REL_DEPTH = 1


def choose_pcfg_rule(rules, rng):
    total = sum(p for _, p in rules)
    assert abs(total - 1.0) < 1e-9, (rules, total)
    r = rng.random()
    c = 0.0
    for name, p in rules:
        c += p
        if r < c:
            return name
    return rules[-1][0]


def validate_pcfg():
    for nt, rules in PCFG_RULES.items():
        total = sum(p for _, p in rules)
        assert abs(total - 1.0) < 1e-9, (nt, total)
        assert all(p >= 0 for _, p in rules)
    print("PCFG rule probabilities validated:")
    for nt, rules in PCFG_RULES.items():
        print(f"  {nt}: " + ", ".join(f"{rhs}={p:.2f}" for rhs, p in rules))

validate_pcfg()

@dataclass
class NPFeatures:
    gender: str
    number: str
    stem: str
    noun_pos: int = -1


def emit_det(lang, gender, number, emit):
    spec = LANG_SPEC[lang]
    d = DETS[lang]
    if spec['det_marks_gender'] and spec['det_fuses_number']:
        emit(d[(gender, number)], 'det', gender=gender, number=number)
    elif spec['det_marks_gender']:
        emit(d[gender], 'det', gender=gender, number=number)
        emit(NUMS[lang][number], 'num', number=number)
    else:
        emit(d, 'det', gender=None, number=None)
        emit(NUMS[lang][number], 'num', number=number)


def emit_adj(lang, gender, stem, spec, rng, emit, subcomp=None):
    lemma_idx = rng.randrange(N_ADJ_LEMMAS)
    emit(ADJS[lang][gender][lemma_idx],
         subcomp or ('adj_agree_pre' if spec['adj_pos'] == 'pre' else 'adj_agree'),
         target_gender=gender, lemma=lemma_idx, stem=stem)


def sample_np(lang, rng, role, pp_depth, rel_depth, emit):
    spec = LANG_SPEC[lang]
    stem = rng.choice(NOUN_STEMS)
    gender = GENDER_TABLE[lang][stem]
    number = rng.choice(['SG', 'PL'])
    noun_tok = noun_surface(lang, stem, role)

    # Deterministic morphology, conditioned on the latent features.
    emit_det(lang, gender, number, emit)

    shape = choose_pcfg_rule(PCFG_RULES['NP_HEAD'], rng)
    has_adj = shape == 'with_adj'

    # The key v10 structural manipulation: exactly the SAME PCFG probabilities
    # are used in every language; adjective position changes only where the
    # already-sampled agreement token is realized.
    if has_adj and spec['adj_pos'] == 'pre':
        emit_adj(lang, gender, stem, spec, rng, emit, subcomp='adj_agree_pre')

    noun_pos = len(pp_depth[1])
    emit(noun_tok, 'n_after_det', gender=gender, number=number, stem=stem)
    npf = NPFeatures(gender=gender, number=number, stem=stem, noun_pos=noun_pos)

    mod = choose_pcfg_rule(PCFG_RULES['MOD'], rng)
    if mod == 'pp' and pp_depth[0] >= MAX_PP_DEPTH:
        mod = 'none'
    if mod == 'rel' and rel_depth[0] >= MAX_REL_DEPTH:
        mod = 'none'

    # Structural axis: all core experimental adjectives are post-nominal, but an
    # adjective can be early (immediately after N) or late (after the same
    # independently sampled modifier chain). This changes only the agreement
    # site's distance from its head, not the PCFG probabilities.
    if has_adj and spec.get('adj_timing', 'early') == 'early':
        emit_adj(lang, gender, stem, spec, rng, emit, subcomp='adj_agree')

    if mod == 'pp':
        sample_pp(lang, rng, pp_depth, rel_depth, emit)
    elif mod == 'rel':
        sample_rel(lang, rng, npf, pp_depth, rel_depth, emit)

    if has_adj and spec.get('adj_timing', 'early') == 'late':
        emit_adj(lang, gender, stem, spec, rng, emit, subcomp='adj_agree')

    return npf


def sample_pp(lang, rng, pp_depth, rel_depth, emit):
    emit(rng.choice(PREPS[lang]), 'structural')
    pp_depth[0] += 1
    sample_np(lang, rng, 'obj', pp_depth, rel_depth, emit)
    pp_depth[0] -= 1


def sample_rel(lang, rng, head, pp_depth, rel_depth, emit):
    spec = LANG_SPEC[lang]
    r = RPROS[lang]
    emit(r[head.gender] if isinstance(r, dict) else r, 'rel_pro',
         target_gender=head.gender if isinstance(r, dict) else None)
    rel_depth[0] += 1
    forms = VINTR[lang]
    feats = spec['verb_agree']
    lemma_idx = rng.randrange(N_VINTR_LEMMAS)
    if feats == ():
        key = '__inv__'
    elif feats == ('number',):
        key = head.number
    else:
        key = (head.gender, head.number)
    tok = forms[key][lemma_idx]
    emit(tok, 'v_agree_rel', target_number=head.number, target_gender=head.gender if isinstance(key, tuple) else None,
         head_stem=head.stem)
    rel_depth[0] -= 1


def pick_verb_form(lang, agree_npf, rng, transitive):
    spec = LANG_SPEC[lang]
    forms_table = VTR[lang] if transitive else VINTR[lang]
    n_lemmas = N_VTR_LEMMAS if transitive else N_VINTR_LEMMAS
    lemma_idx = rng.randrange(n_lemmas)
    feats = spec['verb_agree']
    if feats == ():
        key = '__inv__'
    elif feats == ('number',):
        key = agree_npf.number
    else:
        key = (agree_npf.gender, agree_npf.number)
    tok = forms_table[key][lemma_idx]
    subcomp = 'v_agree' if spec['verb_agree_target'] != 'object' else 'v_agree_obj'
    return tok, subcomp, key


def sample_clause(lang, rng, pp_depth, rel_depth, emit):
    spec = LANG_SPEC[lang]
    transitive = choose_pcfg_rule(PCFG_RULES['CLAUSE'], rng) == 'transitive'

    if spec['word_order'] == 'SVO':
        subj = sample_np(lang, rng, 'subj', pp_depth, rel_depth, emit)
        agree_target = subj
        tok, subcomp, key = pick_verb_form(lang, agree_target, rng, transitive)
        emit(tok, subcomp, target_number=agree_target.number,
             target_gender=agree_target.gender if isinstance(key, tuple) else None,
             agree_source='subject')
        if transitive:
            sample_np(lang, rng, 'obj', pp_depth, rel_depth, emit)
    else:
        subj = sample_np(lang, rng, 'subj', pp_depth, rel_depth, emit)
        obj = sample_np(lang, rng, 'obj', pp_depth, rel_depth, emit) if transitive else None
        agree_target = obj if (spec['verb_agree_target'] == 'object' and obj is not None) else subj
        tok, subcomp, key = pick_verb_form(lang, agree_target, rng, transitive)
        emit(tok, subcomp, target_number=agree_target.number,
             target_gender=agree_target.gender if isinstance(key, tuple) else None,
             agree_source=('object' if agree_target is obj else 'subject'))

    emit(DOT, 'structural')


def sample_sentence(lang, rng):
    tokens, annots = [], []
    def emit(tok, subcomp, **extra):
        tokens.append(tok)
        ann = {'subcomp': subcomp}
        ann.update(extra)
        annots.append(ann)

    pp_depth = [0, tokens]
    rel_depth = [0]
    sample_clause(lang, rng, pp_depth, rel_depth, emit)
    if choose_pcfg_rule(PCFG_RULES['S'], rng) == 'coord':
        emit(CONJS[lang], 'structural')
        sample_clause(lang, rng, pp_depth, rel_depth, emit)
    return tokens, annots


def sample_packed_sequence(lang, rng, max_len=MAX_LEN):
    tokens = [BOS, LANG_TOK[lang]]
    annots = [{'subcomp': 'bos'}, {'subcomp': 'lang_id'}]
    while len(tokens) < max_len - 1:
        s_tok, s_ann = sample_sentence(lang, rng)
        if len(tokens) + len(s_tok) + 1 > max_len:
            break
        tokens.extend(s_tok)
        annots.extend(s_ann)
    tokens.append(EOS)
    annots.append({'subcomp': 'eos'})
    pad_len = max_len - len(tokens)
    tokens.extend([PAD] * pad_len)
    annots.extend([{'subcomp': 'pad'}] * pad_len)
    ids = [STOI[t] for t in tokens]
    return ids, annots, lang


def pcfg_empirical_audit(n=50000, seed=77):
    rng = random.Random(seed)
    s_counts = Counter()
    clause_counts = Counter()
    np_counts = Counter()
    mod_counts = Counter()
    for _ in range(n):
        s_counts[choose_pcfg_rule(PCFG_RULES['S'], rng)] += 1
        clause_counts[choose_pcfg_rule(PCFG_RULES['CLAUSE'], rng)] += 1
        np_counts[choose_pcfg_rule(PCFG_RULES['NP_HEAD'], rng)] += 1
        mod_counts[choose_pcfg_rule(PCFG_RULES['MOD'], rng)] += 1
    print("PCFG empirical audit (target -> observed):")
    for counts, rules in ((s_counts, PCFG_RULES['S']), (clause_counts, PCFG_RULES['CLAUSE']),
                          (np_counts, PCFG_RULES['NP_HEAD']), (mod_counts, PCFG_RULES['MOD'])):
        total = sum(counts.values())
        print('  ' + ', '.join(f"{name}={counts[name]/total:.3f} ({p:.2f})" for name, p in rules))

pcfg_empirical_audit()


# ══════════════════════════════════════════════════════════════════════


## Pre-training audit logic

Two distinct questions need to be separated:

1. **Is the data-generating process behaving as specified?**
2. **Does the model's capacity curve respond to the intended linguistic manipulation?**

The first is addressed by rule normalization, twin invariants, and the leakage audit. None of these validates the theory; they establish that a negative or positive model result is not obviously caused by a sampler bug.


## 5. Leakage audit

The audit targets accidental statistical dependence in adjective agreement. For the relevant samples it compares the observed successive-agreement match rate with the rate expected from the empirical marginal class distribution.

This is deliberately narrower than a general independence proof. The useful interpretation is: if the audit fails, the corpus generator is introducing a correlation that can make agreement prediction easier for reasons unrelated to the intended structure. If it passes, we have cleared one important confound; the lexical and architectural shortcut space still needs to be considered separately.


In [ ]:
def leakage_audit(lang, n=20000, seed=13):
    """
    Compares the observed same-gender rate across two successive ADJ
    targets against the CHANCE rate implied by that language's OWN
    empirical gender marginal -- not a fixed 0.5 (or 0.25 for E). This
    matters because a hamming-flip table need not preserve the anchor's
    class balance; the expected match rate is therefore computed from the
    language's own empirical marginal rather than assuming uniform classes.
    """
    rng = random.Random(seed)
    pairs, all_genders = [], []
    for _ in range(n):
        ids, ann, _ = sample_packed_sequence(lang, rng, max_len=MAX_LEN)
        gs = [a.get('target_gender') for a in ann
              if a['subcomp'] in ('adj_agree', 'adj_agree_pre') and a.get('target_gender')]
        all_genders.extend(gs)
        if len(gs) >= 2:
            pairs.append((gs[0], gs[1]))
    if len(pairs) < 200:
        print(f"[{lang}] too few multi-ADJ sequences ({len(pairs)}) for a leakage check -- skipping.")
        return
    marg = Counter(all_genders)
    total = sum(marg.values())
    chance_same = sum((c / total) ** 2 for c in marg.values())  # sum_g P(g)^2
    same = sum(1 for a, b in pairs if a == b)
    frac = same / len(pairs)
    se = math.sqrt(max(chance_same * (1 - chance_same), 1e-9) / len(pairs))
    z = (frac - chance_same) / se if se > 0 else 0.0
    print(f"[{lang}] observed P(match)={frac:.3f}  expected-under-independence={chance_same:.3f}  "
          f"(marginal={dict(marg)})  z={z:.2f}  n={len(pairs)}")
    assert abs(z) < 4.0, f"Possible leakage in {lang}: z={z:.2f} (observed {frac:.3f} vs expected {chance_same:.3f})"

print("Running leakage audits (expect empirical independence under each language's own marginal)...")
for L in LANGS:
    leakage_audit(L, n=8000)


# ══════════════════════════════════════════════════════════════════════


## 6. Tiny GPT architecture

The architecture is kept intentionally small, but the experimental point is more specific than “smaller model vs larger model.” `n_embd_tok` is a **narrow token-identity channel** that is projected into the wider residual stream. The sweep therefore changes the representational bandwidth available for retaining lexical identity while allowing the downstream transformer computation to remain comparatively wider.

The theoretical interpretation is deliberately modest: we are not identifying a neural subnetwork with `f*` or `g(L)`. We are asking whether increasing the capacity available for lexical identity changes the relative learning curves of carefully matched languages.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        mask = torch.tril(torch.ones(max_len, max_len)).view(1, 1, max_len, max_len)
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(hd)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    """
    n_embd_tok is the TOKEN EMBEDDING dimension -- the channel a
    (language, noun-identity) -> gender lookup has to be smuggled
    through. n_embd is the RESIDUAL/attention/MLP width everything else
    runs in. These were the same number in the first run, which is
    exactly why the lookup table was free: a wide residual stream gave
    the model plenty of room to memorize per-token gender regardless of
    how "small" the model looked by param count. A narrow tok_emb
    projected up into a wider residual stream still has to pass through
    that narrow bottleneck to get noun identity into the computation at
    all -- so THIS is the knob that should actually create a shared
    phase, not overall param count.
    """
    def __init__(self, vocab_size, n_embd, n_layer, n_head, max_len, n_embd_tok=None):
        super().__init__()
        n_embd_tok = n_embd_tok or n_embd
        self.tok_emb = nn.Embedding(vocab_size, n_embd_tok)
        self.tok_proj = (nn.Linear(n_embd_tok, n_embd, bias=False)
                          if n_embd_tok != n_embd else nn.Identity())
        self.pos_emb = nn.Parameter(torch.zeros(1, max_len, n_embd))
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.max_len = max_len

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_proj(self.tok_emb(idx)) + self.pos_emb[:, :T, :]
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        return self.head(x)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

# Capacity grid: (n_layer, n_embd, n_head, n_embd_tok).
#
# n_embd_tok is the new, deliberately-narrow knob (see TinyGPT docstring)
# -- it's listed FIRST in spirit because it's the one doing the actual
# bottlenecking now. n_embd/n_head still vary too, both because the
# residual stream needs *some* room to do the grammar's other work
# (case, verb agreement, structure -- this is a genuinely demanding
# grammar, not a single-task toy) and because the theory's capacity
# variable was never meant to be "one specific tensor's width" -- but
# n_embd_tok is what determines whether the (lang, noun) -> gender table
# is cheap or expensive to store.
#
# The first run swept n_embd from 64 (178K params) up and got acc=1.000
# EVERYWHERE at every point -- confirming the lookup table is cheap for
# any n_embd_tok that large. This grid adds several points where
# n_embd_tok is small enough that memorizing 8 languages x 160 nouns'
# worth of gender assignments should compete directly against everything
# else the token embedding has to encode (noun's OTHER properties:
# nothing else, in this grammar -- but also against fitting 8 languages'
# worth of distinct token identities through the same narrow channel).
#
# Widen/narrow to taste -- the point is to bracket the predicted N*
# crossovers, not to match any particular real-model scale. Kept
# deliberately modest on n_embd/n_layer at the top end -- attention is
# O(T^2) in MAX_LEN and cost scales worse than linearly with depth x
# width jointly, so the last point or two can dominate total wall-clock
# time even though it's "one more entry in a list."
CAPACITY_GRID = [
    (1,  32,  1,   2),
    (1,  32,  1,   4),
    (1,  32,  1,   6),
    (1,  32,  1,   8),
    (2,  64,  2,  12),
    (2,  64,  2,  16),
    (2, 128,  4,  24),
    (2, 128,  4,  32),
    (4, 128,  4,  48),
    (4, 256,  8,  64),
    (6, 256,  8,  96),
]


# ══════════════════════════════════════════════════════════════════════


## 7. Runtime configuration and isolation

This section is bookkeeping rather than theory. The dedicated run directory prevents checkpoints/results from different notebook revisions from being mixed.

`FAST_MODE` is useful for checking the pipeline, but the inferential results should come from the full configuration and multiple seeds. The distinction matters because the capacity curves can be noisy near a threshold; a fast run is a smoke test, not a substitute for the final sweep.


In [ ]:
# v10.1 runtime configuration and strict artifact isolation.

from pathlib import Path

RUN_ID = "pcfg_v10_1"
RUN_DIR = Path("./pcfg_v10_1_runs")
CKPT_DIR = RUN_DIR / "checkpoints"
CACHE_DIR = RUN_DIR / "corpus_cache"
RESULTS_DIR = RUN_DIR / "results"
LOG_DIR = RUN_DIR / "logs"

for _d in (RUN_DIR, CKPT_DIR, CACHE_DIR, RESULTS_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)


def assert_isolated_path(path):
    path = Path(path).resolve()
    root = RUN_DIR.resolve()
    if root not in path.parents and path != root:
        raise RuntimeError(f"Refusing non-v10.1 artifact path: {path}")
    return path


FAST_MODE = True

if FAST_MODE:
    CACHE_EXAMPLES = 50_000
    TRAIN_STEPS = 800
    EVAL_N = 800
    BATCH_SIZE = 1024
    SWEEP_SEEDS = (0,)
else:
    CACHE_EXAMPLES = 200_000
    TRAIN_STEPS = 3000
    EVAL_N = 10_000
    BATCH_SIZE = 1024
    SWEEP_SEEDS = (0, 1, 2)

if device == "cuda":
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True

print(
    f"FAST_MODE={FAST_MODE} | steps={TRAIN_STEPS} | "
    f"seeds={SWEEP_SEEDS} | batch={BATCH_SIZE} | eval={EVAL_N}"
)
print(f"Isolated artifacts: {RUN_DIR.resolve()}")


## 8. Corpus generation and batching

The corpus is generated once from the same sampler and cached as integer token sequences. Training then indexes directly into that cache. This changes throughput, not the distribution of examples.

The generated metadata retains the latent agreement targets so evaluation can be performed at the feature level rather than by inspecting only the surface-token loss.


In [ ]:
# High-throughput training: cached integer corpus + direct tensor batches + CUDA AMP.

def build_cached_corpus(seed, n_examples, langs):
    cache_path = assert_isolated_path(
        CACHE_DIR / f"{RUN_ID}_seed{seed}_n{n_examples}.pt"
    )
    if cache_path.exists():
        return torch.load(cache_path, map_location="cpu", weights_only=True)

    rng = random.Random(100_000 + seed)
    arr = np.empty((n_examples, MAX_LEN), dtype=np.int32)

    for i in range(n_examples):
        lang = rng.choice(langs)
        ids, _, _ = sample_packed_sequence(lang, rng)
        arr[i] = np.asarray(ids, dtype=np.int32)

    tensor = torch.from_numpy(arr)
    torch.save(tensor, cache_path)
    return tensor


TRAIN_CORPUS = {}

def get_train_corpus(seed, langs):
    key = (seed, tuple(langs), CACHE_EXAMPLES)
    if key not in TRAIN_CORPUS:
        print(f"Building/loading cached corpus for seed={seed}...")
        TRAIN_CORPUS[key] = build_cached_corpus(seed, CACHE_EXAMPLES, langs)
        print(f"Corpus shape: {tuple(TRAIN_CORPUS[key].shape)}")
    return TRAIN_CORPUS[key]


def get_random_batch(corpus, batch_size):
    indices = torch.randint(0, corpus.shape[0], (batch_size,))
    batch = corpus[indices]

    if device == "cuda":
        batch = batch.pin_memory().to(device, non_blocking=True)
    else:
        batch = batch.to(device)

    return batch.long()


def train_model(
    n_layer,
    n_embd,
    n_head,
    langs,
    n_embd_tok=None,
    steps=None,
    batch_size=None,
    lr=6e-4,
    seed=0,
    log_every=100,
):
    steps = TRAIN_STEPS if steps is None else steps
    batch_size = BATCH_SIZE if batch_size is None else batch_size

    torch.manual_seed(seed)
    if device == "cuda":
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

    model = TinyGPT(
        VOCAB_SIZE,
        n_embd,
        n_layer,
        n_head,
        MAX_LEN,
        n_embd_tok=n_embd_tok,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    corpus = get_train_corpus(seed, langs)
    pad_id = STOI[PAD]
    use_amp = device == "cuda"

    model.train()
    t0 = time.time()
    tokens_seen = 0

    for step in range(steps):
        ids = get_random_batch(corpus, batch_size)
        x = ids[:, :-1]
        targets = ids[:, 1:]
        targets = targets.masked_fill(targets == pad_id, -100)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
            enabled=use_amp,
        ):
            logits = model(x)
            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                targets.reshape(-1),
                ignore_index=-100,
            )

        loss.backward()
        optimizer.step()

        tokens_seen += ids.numel()

        if step % log_every == 0:
            elapsed = time.time() - t0
            tok_s = tokens_seen / max(elapsed, 1e-6)
            print(
                f"step {step:4d}/{steps} | "
                f"loss={loss.item():.4f} | "
                f"{tok_s/1e6:.2f}M tok/s"
            )

    checkpoint_path = assert_isolated_path(
        CKPT_DIR / f"{RUN_ID}_seed{seed}_dtok{n_embd_tok}.pt"
    )

    torch.save(
        {
            "run_id": RUN_ID,
            "seed": seed,
            "n_layer": n_layer,
            "n_embd": n_embd,
            "n_head": n_head,
            "n_embd_tok": n_embd_tok,
            "vocab_size": VOCAB_SIZE,
            "state_dict": model.state_dict(),
        },
        checkpoint_path,
    )

    print(f"Checkpoint: {checkpoint_path}")
    return model


## 9. Feature-level evaluation

The evaluator aggregates probability over the surface forms associated with each latent class and asks whether the correct class receives the highest aggregate probability. This avoids making the result depend on which particular inflected form happened to be selected as the target.

The two main reported subcomponents have different experimental roles: `adj_agree` is broadly available across the languages, while `v_agree_obj` is specifically informative for the D-family object-agreement contrast. A structurally absent feature is **N/A**, not a failed threshold crossing.


In [ ]:
def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p = successes / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    adj = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return ((centre - adj) / denom, (centre + adj) / denom)

@torch.no_grad()
def feature_accuracy_at_subcomp(model, lang, subcomp, n_eval=4000, batch_size=256, seed=999):
    """
    Restricted-vocab feature accuracy for a gender/class-bearing subcomp
    ('adj_agree', 'adj_agree_pre', 'v_agree', 'v_agree_obj'). Groups all
    valid tokens at that position by their target FEATURE (gender/class)
    and takes argmax over feature groups, not over the full vocabulary.

    Vectorized: matching (batch_row, position) pairs are gathered per
    eval batch and scored in ONE logsumexp call per feature group,
    instead of one torch call + .item() PER MATCHING POSITION. The
    per-position version forces a GPU-CPU sync at every single match
    (hundreds per batch) -- that's what was actually behind the GPU-util
    cliff right after training ended and eval started, not a real
    slowdown in the model or sampler.
    """
    model.eval()
    genders = genders_for(lang)
    if subcomp in ('adj_agree', 'adj_agree_pre'):
        groups = {g: [STOI[t] for t in ADJS[lang][g]] for g in genders}
    elif subcomp in ('v_agree', 'v_agree_obj'):
        spec = LANG_SPEC[lang]
        if spec['verb_agree'] != ('number', 'gender'):
            return None
        groups = {}
        for g in genders:
            toks = []
            for n in ('SG', 'PL'):
                toks += VTR[lang][(g, n)] + VINTR[lang][(g, n)]
            groups[g] = [STOI[t] for t in toks]
    else:
        raise ValueError(subcomp)

    group_names = list(groups.keys())
    gender_to_idx = {g: i for i, g in enumerate(group_names)}
    # Index tensors built once, reused every batch -- no re-allocation per match.
    group_idx_tensors = [torch.tensor(groups[g], dtype=torch.long, device=device) for g in group_names]

    rng = random.Random(seed)
    score_chunks, label_chunks = [], []
    remaining = n_eval
    while remaining > 0:
        bs = min(batch_size, remaining)
        batch_ids, batch_ann = [], []
        for _ in range(bs):
            ids, ann, _ = sample_packed_sequence(lang, rng)
            batch_ids.append(ids)
            batch_ann.append(ann)
        ids_t = torch.tensor(batch_ids, dtype=torch.long, device=device)
        logits = model(ids_t[:, :-1])
        log_probs = F.log_softmax(logits, dim=-1)  # (B, T-1, V)

        # Cheap Python-side scan to find WHICH (row, pred_pos) pairs match --
        # this part is fine (dict lookups, no GPU calls). The GPU work is
        # deferred until every match in this batch is collected.
        rows, cols, labels = [], [], []
        for b, ann in enumerate(batch_ann):
            for pos, a in enumerate(ann):
                if a['subcomp'] != subcomp:
                    continue
                true_g = a.get('target_gender')
                if true_g is None:
                    continue
                pred_pos = pos - 1
                if pred_pos < 0:
                    continue
                rows.append(b)
                cols.append(pred_pos)
                labels.append(gender_to_idx[true_g])

        if rows:
            rows_t = torch.tensor(rows, device=device)
            cols_t = torch.tensor(cols, device=device)
            gathered = log_probs[rows_t, cols_t]  # (M, V) -- one gather, not M gathers
            # One logsumexp call PER GROUP (2 or 4 total), not per match.
            scores = torch.stack(
                [torch.logsumexp(gathered[:, idx], dim=1) for idx in group_idx_tensors], dim=1
            )  # (M, n_groups)
            score_chunks.append(scores.cpu())          # single sync per batch, not per match
            label_chunks.append(torch.tensor(labels))
        remaining -= bs

    if not score_chunks:
        return None
    all_scores = torch.cat(score_chunks, dim=0)
    all_labels = torch.cat(label_chunks, dim=0)
    preds = all_scores.argmax(dim=1)
    n_total = all_labels.numel()
    n_correct = (preds == all_labels).sum().item()
    acc = n_correct / n_total
    lo, hi = wilson_ci(n_correct, n_total)
    return dict(lang=lang, subcomp=subcomp, n=n_total, acc=acc, ci_lo=lo, ci_hi=hi)


def evaluate_capacity_point(model, langs, subcomps=('adj_agree', 'adj_agree_pre', 'v_agree', 'v_agree_obj')):
    results = []
    for lang in langs:
        for sc in subcomps:
            r = feature_accuracy_at_subcomp(model, lang, sc)
            if r is not None:
                results.append(r)
    return results


# ══════════════════════════════════════════════════════════════════════


## 10. Controls and interpretation

The controls are designed to distinguish learnability from the comparative effect of the manipulation. A language that is hard for every capacity is not automatically evidence for a costly shared representation; similarly, a high-accuracy language is not automatically evidence of specialization.

The informative unit is the matched contrast: same or near-same grammar/realization, one controlled intervention, and a corresponding change in the capacity curve. This is why the twin invariants in the language-definition section matter more than absolute accuracy rankings across unrelated languages.


## 11. Single-language control

This helper isolates the underlying task from the multilingual competition. It is useful for diagnosing whether a condition is learnable when language competition is removed.

The control is not itself a test of sharing: single-language training removes the pressure to reuse a common solution. Its main value is identifying whether a surprising multilingual curve is plausibly an optimization/task-difficulty issue rather than a sharing effect.


In [ ]:
def train_control_model(n_layer, n_embd, n_head, lang, n_embd_tok, steps=2000, batch_size=256, seed=0):
    return train_model(n_layer, n_embd, n_head, [lang], n_embd_tok=n_embd_tok,
                       steps=steps, batch_size=batch_size, seed=seed)

# ══════════════════════════════════════════════════════════════════════


## 12. Capacity sweep

This is the main experimental loop. Each configuration is trained with the same language set and evaluated with the same feature-level metrics, while the architecture/capacity configuration changes according to the predefined grid.

For interpretation, retain the distinction between **parameter scaling** and the primary lexical bottleneck `n_embd_tok`. The final summaries index the operational crossover by that latter axis, even though the full configurations also vary depth, residual width, and heads.


In [ ]:
def run_sweep(langs=LANGS, seeds=None, steps=None, batch_size=None):
    seeds = SWEEP_SEEDS if seeds is None else tuple(seeds)
    steps = TRAIN_STEPS if steps is None else steps
    batch_size = BATCH_SIZE if batch_size is None else batch_size

    print(
        f"v10.1 sweep | steps={steps} | batch={batch_size} | "
        f"seeds={seeds} | capacities={len(CAPACITY_GRID)}"
    )

    rows = []

    for n_layer, n_embd, n_head, n_embd_tok in CAPACITY_GRID:
        for seed in seeds:
            print(
                f"\n=== L={n_layer} D={n_embd} H={n_head} "
                f"Dtok={n_embd_tok} seed={seed} ==="
            )

            model = train_model(
                n_layer,
                n_embd,
                n_head,
                langs,
                n_embd_tok=n_embd_tok,
                steps=steps,
                batch_size=batch_size,
                seed=seed,
            )

            n_params = model.num_params()
            evals = evaluate_capacity_point(model, langs)

            for e in evals:
                e.update(
                    n_params=n_params,
                    n_layer=n_layer,
                    n_embd=n_embd,
                    n_head=n_head,
                    n_embd_tok=n_embd_tok,
                    seed=seed,
                    vocab_size=VOCAB_SIZE,
                )
                rows.append(e)

            checkpoint_path = assert_isolated_path(
                RESULTS_DIR / f"{RUN_ID}_rows.json"
            )
            with open(checkpoint_path, "w") as f:
                json.dump(rows, f, indent=2)

            print(
                f"  checkpointed {len(rows)} rows -> {checkpoint_path}"
            )

            del model
            if device == "cuda":
                torch.cuda.empty_cache()

    return rows


## 13. Operational `N*`

The theoretical `N*` is a crossover in competing implementation costs. The notebook cannot observe that internal competition directly, so it defines an empirical proxy: the smallest swept `n_embd_tok` at which mean feature accuracy reaches 0.80.

This proxy is useful for comparing matched curves, but it should not be substituted into the theoretical equation as if it were a direct estimate of the derivation. In particular, `> max swept` is censoring by the experimental grid, not evidence for an infinite crossover.

For the final analysis, the curve itself is more informative than the thresholded scalar: a threshold can hide non-monotonicity and seed variation.


In [ ]:
def crossover_capacity(rows, lang, subcomp, threshold=0.80, key='n_embd_tok'):
    """
    Empirical N* proxy: smallest evaluated capacity (by `key` -- default
    n_embd_tok, the narrow token-identity channel that actually gates the
    (lang, noun) -> gender lookup; use key='n_params' for the old,
    coarser total-parameter view) at which mean accuracy across seeds
    first exceeds `threshold`. Returns (capacity, accs_at_each_capacity)
    or (None, accs) if never crossed within the swept grid (consistent
    with the theory's own N*->infinity edge cases -- report this as a
    real result, not a failure).
    """
    by_capacity = defaultdict(list)
    for r in rows:
        if r['lang'] == lang and r['subcomp'] == subcomp:
            if key not in r:
                raise KeyError(
                    f"Row missing '{key}' (lang={lang}, subcomp={subcomp}) -- this means an "
                    f"old-format checkpoint row slipped through. Re-run with a checkpoint that "
                    f"only contains rows written by the current run_sweep (it now filters these "
                    f"out automatically), rather than silently keying by a different field.")
            by_capacity[r[key]].append(r['acc'])
    if not by_capacity:
        return None, {}
    means = {n: sum(v) / len(v) for n, v in sorted(by_capacity.items())}
    for n, m in means.items():
        if m >= threshold:
            return n, means
    return None, means

def summarize_twin_pairs(rows, subcomp='adj_agree', threshold=0.80, key='n_embd_tok'):
    print(f"\n--- N* summary at subcomp='{subcomp}', threshold={threshold}, capacity axis='{key}' ---")
    for base, twin, axis in TWIN_PAIRS:
        n_base, means_base = crossover_capacity(rows, base, subcomp, threshold, key=key)
        n_twin, means_twin = crossover_capacity(rows, twin, subcomp, threshold, key=key)
        print(f"\n[{axis}]")
        print(f"  {base:6s} N* = {n_base if n_base else '>max swept'}   "
              f"accs by {key}: " + ", ".join(f"{k}:{v:.2f}" for k, v in means_base.items()))
        print(f"  {twin:6s} N* = {n_twin if n_twin else '>max swept'}   "
              f"accs by {key}: " + ", ".join(f"{k}:{v:.2f}" for k, v in means_twin.items()))
        if n_base and n_twin:
            direction = "twin separates LATER (higher N*)" if n_twin > n_base else \
                        "twin separates EARLIER (lower N*)" if n_twin < n_base else \
                        "same N*"
            print(f"  -> {direction}")


# ══════════════════════════════════════════════════════════════════════


## 14. Dependency and isolation validation

This is a reproducibility guard. It checks that the notebook has the expected definitions in place and that generated artifacts belong to the current run namespace.

A successful check establishes internal consistency of the notebook assembly; it is not a scientific validation of the grammar or hypothesis.


In [ ]:
# Notebook dependency / isolation validation — this cell does NOT train anything.
assert 'MAX_LEN' in globals()
assert 'CAPACITY_GRID' in globals()
assert 'LANGS' in globals()
assert 'sample_packed_sequence' in globals()
assert 'CausalSelfAttention' in globals()
assert 'Block' in globals()
assert 'TinyGPT' in globals()
assert 'train_model' in globals()
assert 'evaluate_capacity_point' in globals()
assert 'run_sweep' in globals()
assert RUN_ID == 'pcfg_v10_1'
assert_isolated_path(CKPT_DIR / 'test.pt')
print('✓ All required definitions/configuration are loaded in dependency order.')
print(f'✓ MAX_LEN={MAX_LEN}; capacities={len(CAPACITY_GRID)}; vocab={VOCAB_SIZE}')
print(f'✓ Fast mode: {FAST_MODE} | steps={TRAIN_STEPS} | seeds={SWEEP_SEEDS} | batch={BATCH_SIZE}')
print(f'✓ Isolated artifact root: {RUN_DIR.resolve()}')


## 15. Launch and interpretation

This cell is the experimental entry point. The final summaries report the operational `N*` for the selected feature-level metrics.

When reading them, compare **matched manipulations** rather than ranking all languages. The most important current result to inspect is whether C/Cτ respond differently when the local determiner cue is introduced, and whether the D/Dlo curves move in the theoretically intended direction.

There is an important limitation to carry into any mentor discussion: the present D/Dlo Hamming manipulation changes the noun→class mapping itself. Consequently it can affect both the cost of using the shared/anchor representation and the cost of the language-specific lookup. A D/Dlo threshold difference therefore cannot yet be cleanly interpreted as an ε estimate. The next ε revision should hold native lexical difficulty fixed while changing only the information content of the shared route.

Also, `v_agree_obj` is only defined for languages that actually instantiate object-gender agreement. The absence of that feature should be rendered as N/A in interpretation, not as `> max swept`.


In [ ]:
# Run the experiment explicitly from this cell.
#
# FAST_MODE=True:
#   800 steps, 1 seed, 50k cached examples, 800 eval examples.
#
# FAST_MODE=False:
#   3000 steps, 3 seeds, 200k cached examples, 10k eval examples.

print(
    f"Starting v10.1 | FAST_MODE={FAST_MODE} | "
    f"steps={TRAIN_STEPS} | seeds={SWEEP_SEEDS} | "
    f"batch={BATCH_SIZE}"
)

rows = run_sweep()

results_path = assert_isolated_path(
    RESULTS_DIR / f"{RUN_ID}_sweep.json"
)
with open(results_path, "w") as f:
    json.dump(rows, f, indent=2)

print(f"Saved: {results_path}")

summarize_twin_pairs(
    rows,
    subcomp="adj_agree",
    threshold=0.80,
)

summarize_twin_pairs(
    rows,
    subcomp="v_agree_obj",
    threshold=0.80,
)
